# Turkish Legal RAG Demo - BGE-M3 Hybrid + Qwen2.5-7B LoRA

Bu notebook demo içindir:

```text
Kullanıcı sorusu
-> BGE-M3 + BM25 hybrid retriever
-> top legal contexts
-> Qwen2.5-7B-Instruct + LoRA adapter
-> grounded Turkish legal answer with citation
```

Final retrieval ayarları:

```text
alpha = 0.70
dense_candidates = 300
bm25_candidates = 100
preliminary_top_k = 50
```

Not: LoRA adapter klasörünün Kaggle dataset'e eklenmiş olması gerekir. Notebook `adapter_config.json` dosyasını otomatik arar.

In [ ]:
!nvidia-smi

## 1. Ortam Ayarları ve Paketler

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib.util
import subprocess
import sys

required = {
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "huggingface_hub": "huggingface-hub",
}
missing = [pip_name for module_name, pip_name in required.items() if importlib.util.find_spec(module_name) is None]
print("Missing packages:", missing)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

## 2. Dosyaları Working Klasörüne Kopyala

In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/legal-rag")

for path in [
    WORK_DIR / "scripts",
    WORK_DIR / "data/processed",
    WORK_DIR / "data/index",
    WORK_DIR / "data/eval",
]:
    path.mkdir(parents=True, exist_ok=True)

def find_input_file(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Kaggle dataset'e ekledin mi?")
    return matches[0]

def copy_required(name: str, dest_dir: Path) -> Path:
    src = find_input_file(name)
    dst = dest_dir / name
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
    return dst

for name in [
    "rag_answer.py",
    "evaluate_retrieval.py",
]:
    copy_required(name, WORK_DIR / "scripts")

for name in ["retrieval_corpus.json", "retrieval_chunks.json"]:
    copy_required(name, WORK_DIR / "data/processed")

for name in ["faiss_bge_m3.index", "metadata_bge_m3.json", "index_config_bge_m3.json"]:
    copy_required(name, WORK_DIR / "data/index")

print("\nWorking files:")
!find /kaggle/working/legal-rag -maxdepth 4 -type f | sort

## 3. LoRA Adapter Klasörünü Bul

In [ ]:
from pathlib import Path

adapter_configs = sorted(INPUT_ROOT.rglob("adapter_config.json"))
print("Found adapter configs:")
for path in adapter_configs:
    print("-", path)

if not adapter_configs:
    raise FileNotFoundError("adapter_config.json bulunamadı. final_600 LoRA klasörünü Kaggle dataset'e eklemen gerekiyor.")

# Birden fazla adapter varsa burada doğru olanı elle seçebilirsin.
ADAPTER_PATH = adapter_configs[0].parent
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

print("Using adapter:", ADAPTER_PATH)
print("Base model:", BASE_MODEL)
!ls -lh "$ADAPTER_PATH"

## 4. RAG Sistemini Yükle

In [ ]:
import sys
import torch

sys.path.insert(0, str(WORK_DIR / "scripts"))
from rag_answer import LegalRAG

rag = LegalRAG(
    index_path=WORK_DIR / "data/index/faiss_bge_m3.index",
    metadata_path=WORK_DIR / "data/index/metadata_bge_m3.json",
    config_path=WORK_DIR / "data/index/index_config_bge_m3.json",
    corpus_path=WORK_DIR / "data/processed/retrieval_corpus.json",
    alpha=0.70,
    dense_candidates=300,
    bm25_candidates=100,
    preliminary_top_k=50,
)

# Embedding modelini CPU'da tutuyoruz; Qwen 7B için GPU VRAM kalsın.
# Tek soru demo/eval için CPU embedding yeterli.
rag.load_retriever(embedding_device="cpu")

rag.load_llm(
    base_model=BASE_MODEL,
    adapter_path=ADAPTER_PATH,
    load_in_4bit=True,
)

print("RAG ready.")

## 5. Tek Soru Demo

In [ ]:
question = "İşçi 2 gün işe gelmezse ne olur?"

result = rag.answer(
    question,
    top_k=5,
    max_new_tokens=384,
    do_sample=False,
)

print("SORU:")
print(result["question"])
print("\nCEVAP:")
print(result["answer"])

print("\nKAYNAKLAR:")
for ctx in result["contexts"]:
    print(f"[{ctx['rank']}] score={ctx['score']:.4f} | {ctx['citation']}")
    print(f"    parent_id: {ctx['parent_id']}")

## 6. Birkaç Test Sorusu

In [ ]:
test_questions = [
    "Kişisel veriler yurt dışına hangi şartlarda aktarılır?",
    "Birini öldürmek suç mudur?",
    "Kiracı depozitoyu hangi şartlarda geri alabilir?",
    "Yıllık ücretli izin hakkından vazgeçilebilir mi?",
]

for question in test_questions:
    print("\n" + "=" * 120)
    print("SORU:", question)
    print("=" * 120)
    result = rag.answer(question, top_k=5, max_new_tokens=384, do_sample=False)
    print(result["answer"])
    print("\nKaynaklar:")
    for ctx in result["contexts"][:3]:
        print(f"- {ctx['citation']} ({ctx['parent_id']})")

## 7. Base Model ile Karşılaştırma İstersen

In [ ]:
# Bu hücre opsiyonel. Fine-tuned LoRA modeli unload edip base modeli yükler.
# GPU memory hatası alırsan session restart edip bu hücreyi tek başına çalıştırmak daha güvenli.
#
# import gc, torch
# del rag.model
# rag.model = None
# gc.collect()
# torch.cuda.empty_cache()
#
# rag.load_llm(
#     base_model=BASE_MODEL,
#     adapter_path=None,
#     load_in_4bit=True,
# )
# result_base = rag.answer("İşçi 2 gün işe gelmezse ne olur?", top_k=5, max_new_tokens=384, do_sample=False)
# print(result_base["answer"])

## 8. Demo Notları

Eğer cevaplar çok kısa kalırsa:

```python
max_new_tokens=512
```

Eğer model çok yavaşsa:

```python
top_k=3
```

Eğer CUDA OOM alırsan:

- Notebook session'ı restart et.
- Önce sadece fine-tuned LoRA demo hücrelerini çalıştır.
- Base model karşılaştırma hücresini aynı session'da çalıştırma.